# Drifting ICNN

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import random
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple, List, Sequence

In [2]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

### Hyperparameters

In [ ]:
METHODS = ["kernel", "ot_direct", "npf"]
NUM_ITERS = 3000
BATCH_SIZE = 128
LATENT_DIM = 64
NOISE_DIM = 64
LR_GEN = 2e-4
LR_VAE = 1e-3
# Inner-loop step count for the NPF drift. 15 (vs 5 for the original
# plain ICNN) because the NPF has per-layer convex quadratic injections in
# addition to the non-negative cascade — more parameters → needs more
# inner Adam descent before each generator update.
INNER_STEPS = 15
VAE_WARMUP_STEPS = 400
SEED = 42
seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MNIST_ROOT = "./data"
SAVE_PATH = "results/mnist/mnist_drifting_results.png"

# NPF (Vesseron & Cuturi, 2024) ICNN parameterization replaces the plain
# ICNN drift field. Hidden widths are kept identical to the previous ICNN
# baseline so capacity is comparable.
NPF_HIDDEN_DIMS = [512, 256, 128, 64] # [128, 128, 128, 128]
# NPF outer/inner low-rank PSD ranks.
NPF_OUTER_RANK = 4
NPF_INNER_RANK = 1
# Activation: softplus is the canonical ICNN activation in the proposal.
# We deliberately avoid ELU here because ELU(u)=u for u≥0 makes ELU''=0 in
# the positive regime — at init the cascade input lies right there, so the
# cascade would contribute zero curvature to ψ and the inner loop could
# only fit a *linear* transport map, defeating the NPF parameterization.
# softplus(βu)/β has f''=β·σ(βu)(1−σ(βu)) > 0 everywhere; β=1 gives a
# broad curved band that overlaps the small-magnitude operating regime.
NPF_ACTIVATION = "softplus"
NPF_ELU_ALPHA = 1.0
NPF_SOFTPLUS_BETA = 1.0  # was 20 (too ReLU-like, narrow curvature band)
# init_eps controls the live-at-init perturbation magnitude. 1e-2 (vs the
# 1e-3 default elsewhere) gives the LogNormal cascade enough "warmth" that
# Adam can escape the symmetric basin in a few inner steps; the resulting
# departure from identity is still O(1e-2) per-coordinate, well within the
# proposal's "T_ω(x) ≈ x at t=0" tolerance.
NPF_INIT_EPS = 1e-2

# Adam hyperparameters for the NPF inner loop. The proposal's inner step
# is L2-regression of T_omega(x) against the Sinkhorn barycentric target;
# Adam handles the strongly anisotropic gradient scales of the LogNormal
# cascade. lr=3e-3 (vs default 1e-3) compensates for the small magnitudes
# of cascade gradients at near-identity init.
NPF_INNER_LR = 3e-3
NPF_ADAM_BETAS = (0.9, 0.999)
NPF_ADAM_EPS = 1e-8
NPF_WEIGHT_DECAY = 0.0
NPF_GRAD_CLIP = 5.0

# Asymmetric generator widths. The output is still LATENT_DIM.
GEN_HIDDEN_DIMS = [512, 256, 128]

# Sinkhorn regularization is now dimensionless because we normalize the cost matrix.
# Tune this rather than relying on the common 0.05 default.
SINKHORN_REG = 0.2
SINKHORN_REG_GRID = [0.03, 0.05, 0.1, 0.2, 0.5, 1.0]
SINKHORN_NUM_ITERS = 80
NORMALIZE_OT_COST = True


### 1. The VAE (Encoder/Decoder)
We need this to map Images <-> Latent Space. 
The Drifting Model operates ONLY in the latent space.

In [4]:
class MNISTVAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 14 -> 7
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# 7 -> 4
            nn.SiLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.SiLU(),
            nn.Linear(256, 128 * 4 * 4),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=0), # 4 -> 7
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),                     # 7 -> 14
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),                      # 14 -> 28
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor):
        h = self.decoder_fc(z)
        return self.decoder(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar



class LatentGenerator(nn.Module):
    """
    Asymmetric latent generator.

    With hidden_dims=(512, 256, 128), the Linear stack is:
        NOISE_DIM -> 512 -> 256 -> 128 -> LATENT_DIM
    For the current MNIST setup LATENT_DIM=64, so this gives the requested
    512 -> 256 -> 128 -> 64 progression at the generator output.
    """
    def __init__(self, noise_dim: int = 64, latent_dim: int = 64,
                 hidden_dims: Tuple[int, ...] = (512, 256, 128)):
        super().__init__()
        dims = [noise_dim, *hidden_dims, latent_dim]
        layers = []
        for in_dim, out_dim in zip(dims[:-2], dims[1:-1]):
            layers += [nn.Linear(in_dim, out_dim), nn.SiLU()]
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.net = nn.Sequential(*layers)
        self._init_output_scale(target_std=1.0)

    def _init_output_scale(self, target_std: float):
        with torch.no_grad():
            device = self.net[0].weight.device
            test_input = torch.randn(2000, self.net[0].in_features, device=device)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps: torch.Tensor):
        return self.net(eps)


### 2. NPF ICNN & Sinkhorn (Same logic, NPF parameterization of $\psi_\omega$)
The convex potential is now an NPF (Vesseron & Cuturi, 2024) input convex network: a learnable PSD outer base + a deep cascade of per-layer convex quadratic injections combined through non-negative (`exp`-parameterized) hidden-to-hidden weights. Both the principled LogNormal init and the identity init are applied jointly — they are complementary, not alternatives.

In [5]:
def normalized_cost(x: torch.Tensor, y: torch.Tensor,
                    p: int = 2, squared: bool = True,
                    normalize: bool = True):
    """
    Pairwise cost with optional scale normalization.

    In latent spaces, raw ||x-y||² grows with dimension and latent scale.
    Normalizing by the batch median makes Sinkhorn `reg` dimensionless and
    avoids the high-dimensional underflow caused by exp(-C / reg).
    """
    C = torch.cdist(x, y, p=p)
    if squared:
        C = C ** 2
    if normalize:
        scale = C.detach().median().clamp_min(1e-6)
        C = C / scale
    return C


def sinkhorn(C: torch.Tensor, reg: float = 0.2, num_iters: int = 80):
    """
    Stable log-domain Sinkhorn plan for balanced minibatches.

    Returns a plan with approximately uniform marginals. The later row
    normalization used for barycentric targets is still kept because it is
    numerically convenient and works for unequal batch sizes too.
    """
    if reg <= 0:
        raise ValueError("Sinkhorn regularization `reg` must be positive.")

    n, m = C.shape
    log_K = -C / reg
    log_u = torch.zeros(n, device=C.device, dtype=C.dtype)
    log_v = torch.zeros(m, device=C.device, dtype=C.dtype)
    log_a = torch.full((n,), -math.log(n), device=C.device, dtype=C.dtype)
    log_b = torch.full((m,), -math.log(m), device=C.device, dtype=C.dtype)

    for _ in range(num_iters):
        log_u = log_a - torch.logsumexp(log_K + log_v.unsqueeze(0), dim=1)
        log_v = log_b - torch.logsumexp(log_K.transpose(0, 1) + log_u.unsqueeze(0), dim=1)

    log_P = log_u.unsqueeze(1) + log_K + log_v.unsqueeze(0)
    return torch.exp(log_P)


def sinkhorn_barycentric_target(x: torch.Tensor, y: torch.Tensor,
                                reg: float = SINKHORN_REG,
                                num_iters: int = SINKHORN_NUM_ITERS,
                                normalize_cost: bool = NORMALIZE_OT_COST):
    """
    Compute y_bar_i = sum_j P_ij y_j / sum_j P_ij.
    This is shared by `ot_direct` and `icnn` so both use the same regularization.
    """
    C = normalized_cost(x, y, squared=True, normalize=normalize_cost)
    P = sinkhorn(C, reg=reg, num_iters=num_iters)
    P_row = P / (P.sum(1, keepdim=True).clamp_min(1e-8))
    y_target = P_row @ y
    return y_target, C, P


In [ ]:
# NPF (Vesseron & Cuturi, 2024) ICNN potential + drift field — imported
# from `npf_drift.py` so the same module powers this MNIST notebook, the
# gaussian-init ablation, and the width/depth ablation. See
#   * npf_drift.NPFInputConvexPotential — joint principled-LogNormal +
#     identity init contract (NOT a choice; both applied jointly).
#   * npf_drift.NPFDriftField — Adam inner loop with optional Gaussian-OT
#     init and a user-supplied Sinkhorn target function.
#
# We pass a log-domain barycentric Sinkhorn target with median-cost
# normalization (matches `sinkhorn_barycentric_target` from the previous
# cell) so `sinkhorn_reg` is dimensionless even at d=64 latent dim.
from npf_drift import (
    NPFInputConvexPotential,
    NPFDriftField,
    barycentric_target_log,
)


def make_npf_sinkhorn_target_fn(reg: float):
    """Closure over reg for NPFDriftField.sinkhorn_target_fn."""
    return lambda x, y: barycentric_target_log(
        x, y,
        reg=reg,
        num_iters=SINKHORN_NUM_ITERS,
        normalize_cost=NORMALIZE_OT_COST,
    )


### 3. Baselines (Kernel & OT Direct) - Adapted for Latent Space

In [7]:
def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.02, 0.05, 0.2)):
    """
    Compute the drifting field V(x) following the original Drifting Model (Deng et al.)
    Algorithm 2 / Eq. 11 of the paper.

    Returns V (detached), such that the training loss is:
        target = sg(x_gen + V)
        loss = ||x_gen - target||²

    The loss value equals ||V||², which DECREASES as q → p.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    # Targets: [negatives (=generated), positives (=data)]
    targets = torch.cat([x, y], dim=0)  # [N+M, D]

    # ── Pairwise ℓ₂ distances between generated samples and all targets ──
    dist = torch.cdist(x, targets)  # [N, N+M]

    # ── Distance normalization (Appendix A.6) ──
    # Normalize so mean pairwise distance ≈ √D
    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    # ── Self-masking, to prevent trivial zero-distance matches between generated samples and themselves ──
    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    # ── Force accumulation (NO per-temperature normalization) ──
    V = torch.zeros_like(x)

    for tau in tau_list:
        # Sinkhorn-like affinities: a_ij = exp(-d_ij / tau)
        # with double softmax normalization (over rows and columns) to ensure better gradients and prevent collapse
        # The tau (temperature) controls the sharpness of the affinities: smaller tau → sharper affinities, more emphasis on closer targets
        logits = -dist_normed / tau

        # Double softmax (paper's Alg. 2: softmax over y, then over x)
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]                   # negative affinities (to generated samples)
        aff_pos = affinity[:, N:]                   # positive affinities (to data samples)
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        # Drifting coefficients: attract by positives, repel by negatives
        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)  # [N, N+M]

        # Force = weighted combination of aim directions (target - x) vectors
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R  # Raw force, NO normalization

    return V.detach()


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = SINKHORN_REG,
                        num_iters: int = SINKHORN_NUM_ITERS,
                        normalize_cost: bool = NORMALIZE_OT_COST):
    """
    Direct OT displacement V(x) = T_Sinkhorn(x) - x.

    Uses the same normalized-cost/log-domain Sinkhorn target as ICNN, so
    differences between the two methods come from ICNN amortization rather
    than from different OT hyperparameters.
    """
    x = x_gen.detach()
    y = y_pos.detach()
    y_target, _, _ = sinkhorn_barycentric_target(
        x, y,
        reg=reg,
        num_iters=num_iters,
        normalize_cost=normalize_cost,
    )
    return (y_target - x).detach()


### 4. Generator (Latent Space Only)

In [8]:
# Latent generator is defined in Section 1 as `LatentGenerator`.
# Quick sanity-check utility:
def inspect_generator(gen: nn.Module, noise_dim: int = NOISE_DIM, batch_size: int = 512, device: str = DEVICE):
    gen = gen.to(device)
    with torch.no_grad():
        z = gen(torch.randn(batch_size, noise_dim, device=device))
    print(f"Generator latent stats -> mean: {z.mean().item():.4f}, std: {z.std().item():.4f}, shape: {tuple(z.shape)}")

### 5. Training Loop for MNIST

In [ ]:
def _next_batch(data_iter, loader):
    try:
        x_real, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        x_real, _ = next(data_iter)
    return x_real, data_iter


def train_mnist(method='kernel', num_iters=NUM_ITERS, batch_size=BATCH_SIZE,
                lr_gen=LR_GEN, inner_steps=INNER_STEPS, device=DEVICE,
                sinkhorn_reg=SINKHORN_REG):
    device = torch.device(device)

    # 1) Data
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST(root=MNIST_ROOT, train=True, download=True, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    data_iter = iter(loader)

    # 2) VAE warmup for a stable latent manifold
    vae = MNISTVAE(latent_dim=LATENT_DIM).to(device)
    vae_optim = optim.Adam(vae.parameters(), lr=LR_VAE)
    print(f"[{method}] Warming up VAE for {VAE_WARMUP_STEPS} steps...")
    vae.train()
    for _ in range(VAE_WARMUP_STEPS):
        x_real, data_iter = _next_batch(data_iter, loader)
        x_real = x_real.to(device, non_blocking=True)

        recon, mu, logvar = vae(x_real)
        rec_loss = F.binary_cross_entropy(recon, x_real)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        vae_loss = rec_loss + 1e-3 * kl_loss

        vae_optim.zero_grad(set_to_none=True)
        vae_loss.backward()
        vae_optim.step()
    print(f"[{method}] VAE warmup done.")

    # 3) Latent generator + optional NPF drift model
    gen = LatentGenerator(
        noise_dim=NOISE_DIM,
        latent_dim=LATENT_DIM,
        hidden_dims=tuple(GEN_HIDDEN_DIMS),
    ).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr_gen)

    npf_drift = None
    if method == 'npf':
        npf_drift = NPFDriftField(
            dim=LATENT_DIM,
            hidden_dims=NPF_HIDDEN_DIMS,
            outer_rank=NPF_OUTER_RANK,
            inner_rank=NPF_INNER_RANK,
            activation=NPF_ACTIVATION,
            elu_alpha=NPF_ELU_ALPHA,
            softplus_beta=NPF_SOFTPLUS_BETA,
            init_eps=NPF_INIT_EPS,
            inner_steps=inner_steps,
            inner_lr=NPF_INNER_LR,
            adam_betas=NPF_ADAM_BETAS,
            adam_eps=NPF_ADAM_EPS,
            weight_decay=NPF_WEIGHT_DECAY,
            grad_clip=NPF_GRAD_CLIP,
            sinkhorn_reg=sinkhorn_reg,
            sinkhorn_iters=SINKHORN_NUM_ITERS,
            # Override the module's direct-domain default with the
            # log-domain, normalized-cost target used elsewhere in this
            # notebook so reg stays dimensionless at d=64.
            sinkhorn_target_fn=make_npf_sinkhorn_target_fn(sinkhorn_reg),
            init_mode="identity",
        ).to(device)
    elif method not in {'kernel', 'ot_direct'}:
        raise ValueError(f"Unknown method: {method}")

    losses, v_norms, snapshots = [], [], []
    vae.eval()
    snapshot_period = max(1, num_iters // 8)

    for it in range(num_iters):
        x_real_img, data_iter = _next_batch(data_iter, loader)
        x_real_img = x_real_img.to(device, non_blocking=True)

        with torch.no_grad():
            mu, _ = vae.encode(x_real_img)
            y_pos = mu  # deterministic latent targets reduce OT noise

        eps = torch.randn(batch_size, NOISE_DIM, device=device)
        x_gen = gen(eps)

        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.5, 1.0, 2.0))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(
                x_gen,
                y_pos,
                reg=sinkhorn_reg,
                num_iters=SINKHORN_NUM_ITERS,
                normalize_cost=NORMALIZE_OT_COST,
            )
        elif method == 'npf':
            V = npf_drift.compute_V(x_gen, y_pos)

        target_pts = (x_gen.detach() + V).detach()
        loss_drift = ((x_gen - target_pts) ** 2).mean()
        v_norm = (V ** 2).mean().item()

        # Keep generated latents near N(0, I) so they stay decodable.
        second_moment = x_gen.pow(2).mean().clamp_min(1e-6)
        latent_reg = 0.5 * (second_moment - torch.log(second_moment))
        loss = loss_drift + 0.01 * latent_reg

        gen_optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % snapshot_period == 0 or it == num_iters - 1:
            with torch.no_grad():
                z_eval = gen(torch.randn(64, NOISE_DIM, device=device))
                x_eval_img = vae.decode(z_eval).cpu()
                snapshots.append((it, x_eval_img.clone()))
            print(
                f"[{method}] Iter {it}/{num_iters} | "
                f"loss={loss.item():.4f} | ||V||^2={v_norm:.5f} | sinkhorn_reg={sinkhorn_reg}"
            )

    return {
        'losses': losses,
        'v_norms': v_norms,
        'snapshots': snapshots,
        'method': method,
        'sinkhorn_reg': sinkhorn_reg,
    }


def run_reg_sweep(method='ot_direct', reg_grid=SINKHORN_REG_GRID,
                  num_iters=500, batch_size=BATCH_SIZE, device=DEVICE):
    """Small regularization sweep. With NORMALIZE_OT_COST=True, these reg
    values are dimensionless."""
    sweep_results = []
    for reg in reg_grid:
        print(f"\n{'='*60}\n{method} with sinkhorn_reg={reg}\n{'='*60}")
        sweep_results.append(
            train_mnist(
                method=method,
                num_iters=num_iters,
                batch_size=batch_size,
                lr_gen=LR_GEN,
                inner_steps=INNER_STEPS,
                device=device,
                sinkhorn_reg=reg,
            )
        )
    return sweep_results


In [10]:
# Find best regularization parameter - using ot_direct bc faster than npf
reg_sweep_results = run_reg_sweep(
    method="ot_direct",
    reg_grid=SINKHORN_REG_GRID,
    num_iters=500,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# Pick the reg whose tail ||V||^2 (last 50 iters) is smallest — the same
# convergence diagnostic plotted/printed elsewhere in the notebook.
def _tail_v_norm(res, tail=50):
    vs = res["v_norms"]
    return float(np.mean(vs[-tail:])) if len(vs) >= tail else float(np.mean(vs))

reg_sweep_summary = sorted(reg_sweep_results, key=_tail_v_norm)
print("\nSinkhorn-reg sweep (sorted by tail ||V||^2):")
for r in reg_sweep_summary:
    print(f"  reg={r['sinkhorn_reg']:>5}  tail||V||^2={_tail_v_norm(r):.5f}")

BEST_SINKHORN_REG = reg_sweep_summary[0]["sinkhorn_reg"]
print(f"\nBest sinkhorn_reg = {BEST_SINKHORN_REG}")


ot_direct with sinkhorn_reg=0.03
[ot_direct] Warming up VAE for 400 steps...
[ot_direct] VAE warmup done.
[ot_direct] Iter 0/500 | loss=1.4031 | ||V||^2=1.39810 | sinkhorn_reg=0.03
[ot_direct] Iter 62/500 | loss=0.7068 | ||V||^2=0.70180 | sinkhorn_reg=0.03
[ot_direct] Iter 124/500 | loss=0.5657 | ||V||^2=0.56058 | sinkhorn_reg=0.03
[ot_direct] Iter 186/500 | loss=0.6279 | ||V||^2=0.62268 | sinkhorn_reg=0.03
[ot_direct] Iter 248/500 | loss=0.5371 | ||V||^2=0.53175 | sinkhorn_reg=0.03
[ot_direct] Iter 310/500 | loss=0.5686 | ||V||^2=0.56315 | sinkhorn_reg=0.03
[ot_direct] Iter 372/500 | loss=0.5549 | ||V||^2=0.54954 | sinkhorn_reg=0.03
[ot_direct] Iter 434/500 | loss=0.4936 | ||V||^2=0.48839 | sinkhorn_reg=0.03
[ot_direct] Iter 496/500 | loss=0.5085 | ||V||^2=0.50310 | sinkhorn_reg=0.03
[ot_direct] Iter 499/500 | loss=0.5110 | ||V||^2=0.50581 | sinkhorn_reg=0.03

ot_direct with sinkhorn_reg=0.05
[ot_direct] Warming up VAE for 400 steps...
[ot_direct] VAE warmup done.
[ot_direct] Iter 0/

In [11]:
# Run Training for all methods
results_mnist = []
for m in METHODS:
    print(f"\n{'='*60}\nTraining MNIST: {m}\n{'='*60}")
    results_mnist.append(
        train_mnist(
            method=m,
            num_iters=NUM_ITERS,
            batch_size=BATCH_SIZE,
            lr_gen=LR_GEN,
            inner_steps=INNER_STEPS,
            device=DEVICE,
            sinkhorn_reg=BEST_SINKHORN_REG,
        )
    )



Training MNIST: kernel
[kernel] Warming up VAE for 400 steps...
[kernel] VAE warmup done.
[kernel] Iter 0/3000 | loss=0.2866 | ||V||^2=0.28162 | sinkhorn_reg=0.5
[kernel] Iter 375/3000 | loss=0.0093 | ||V||^2=0.00383 | sinkhorn_reg=0.5
[kernel] Iter 750/3000 | loss=0.0117 | ||V||^2=0.00622 | sinkhorn_reg=0.5
[kernel] Iter 1125/3000 | loss=0.0075 | ||V||^2=0.00190 | sinkhorn_reg=0.5
[kernel] Iter 1500/3000 | loss=0.0091 | ||V||^2=0.00333 | sinkhorn_reg=0.5
[kernel] Iter 1875/3000 | loss=0.0086 | ||V||^2=0.00299 | sinkhorn_reg=0.5
[kernel] Iter 2250/3000 | loss=0.0108 | ||V||^2=0.00506 | sinkhorn_reg=0.5
[kernel] Iter 2625/3000 | loss=0.0073 | ||V||^2=0.00153 | sinkhorn_reg=0.5
[kernel] Iter 2999/3000 | loss=0.0073 | ||V||^2=0.00154 | sinkhorn_reg=0.5

Training MNIST: ot_direct
[ot_direct] Warming up VAE for 400 steps...
[ot_direct] VAE warmup done.
[ot_direct] Iter 0/3000 | loss=1.3940 | ||V||^2=1.38898 | sinkhorn_reg=0.5
[ot_direct] Iter 375/3000 | loss=0.0373 | ||V||^2=0.03060 | sink

### Visualization

In [12]:
def plot_mnist_results(results_list, save_path=SAVE_PATH):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])

    fig, axes = plt.subplots(n_methods, n_snaps, figsize=(3.5 * n_snaps, 3.5 * n_methods))
    if n_methods == 1:
        axes = np.array([axes])
    if n_snaps == 1:
        axes = axes.reshape(n_methods, 1)

    for row, res in enumerate(results_list):
        for col, (it, imgs) in enumerate(res['snapshots']):
            ax = axes[row, col]
            grid = make_grid(imgs[:16], nrow=4, normalize=True, value_range=(0, 1))
            grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()
            ax.imshow(grid_np, cmap='gray')
            ax.set_title(f"{res['method']} | iter {it}")
            ax.axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved MNIST results to {save_path}")
    plt.close(fig)

if 'results_mnist' in globals() and len(results_mnist) > 0:
    plot_mnist_results(results_mnist)
else:
    print("Run the training cell first to populate `results_mnist`.")

Saved MNIST results to results/mnist/mnist_drifting_results.png
